### **Implementasi Transfer Learning Menggunakan BERT untuk Klasifikasi Artikel Berita pada Dataset AG News.**

**Impor Library dan Pengaturan Awal**

Sel ini memuat seluruh library yang dipakai pada tahap eksplorasi data: `pandas` dan `numpy` untuk pengolahan data, `matplotlib` dan `seaborn` untuk visualisasi, `torch` sebagai kerangka kerja deep learning, serta `datasets` dan `transformers` dari Hugging Face. Keluaran berupa versi PyTorch dan status ketersediaan GPU (CUDA) digunakan untuk memastikan lingkungan siap dipakai.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

from collections import Counter
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("Semua library berhasil diimpor.")
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

**Memuat Dataset AG News**

Fungsi `load_dataset` mengunduh dataset AG News dari Hugging Face Hub dan menyimpannya sebagai objek `DatasetDict` yang berisi split `train` dan `test`. Hasil cetakan menampilkan nama kolom serta jumlah baris pada setiap split.

In [ ]:
dataset = load_dataset("fancyzhx/ag_news")

print(dataset)

**Pemeriksaan Struktur dan Jumlah Data**

Sel ini menampilkan fitur dataset (tipe data kolom `text` dan `label`, termasuk nama kelas), daftar kolom, serta jumlah data latih dan data uji. Informasi ini dipakai untuk memverifikasi bahwa dataset sesuai dengan yang diharapkan sebelum diproses lebih lanjut.

In [ ]:
print("Fitur Dataset:")
print(dataset["train"].features)

print("\nNama Kolom:")
print(dataset["train"].column_names)

print("\nJumlah Data:")
print("Training:", len(dataset["train"]))
print("Testing :", len(dataset["test"]))

**Melihat Contoh Data**

Tiga data pertama dari set latih ditampilkan lengkap dengan label numerik, nama kategori (melalui `int2str`), dan isi artikel. Langkah ini membantu memahami bentuk teks mentah serta pemetaan label ke kategori.

In [ ]:
for i in range(3):

    sample = dataset["train"][i]

    print(f"\nData ke-{i+1}")
    print("Label:", sample["label"])
    print("Kategori:", dataset["train"].features["label"].int2str(sample["label"]))
    print("Isi Berita:", sample["text"])
    print("-" * 80)

**Konversi ke Pandas DataFrame**

Split latih dan uji diubah menjadi `DataFrame` agar analisis eksploratif dapat dilakukan dengan fungsi `pandas`. Metode `head()` menampilkan beberapa baris awal untuk memeriksa hasil konversi.

In [ ]:
df_train = dataset["train"].to_pandas()
df_test = dataset["test"].to_pandas()

display(df_train.head())

**Informasi Umum DataFrame**

Fungsi `info()` menampilkan jumlah baris, tipe data, dan jumlah nilai non-null setiap kolom, sedangkan `shape` menunjukkan ukuran (baris, kolom) data. Hasilnya dipakai untuk memastikan kolom lengkap dan bertipe sesuai.

In [ ]:
print("Informasi Dataset Training:")
df_train.info()

print("\nInformasi Dataset Testing:")
df_test.info()

print("\nUkuran Dataset:")
print("Training:", df_train.shape)
print("Testing :", df_test.shape)

**Distribusi Kelas pada Data Latih**

Sel ini menghitung jumlah dan persentase artikel pada setiap kategori, lalu menyajikannya dalam tabel. Tujuannya mengetahui apakah data seimbang, karena ketidakseimbangan kelas dapat memengaruhi pembelajaran dan pemilihan metrik evaluasi.

In [ ]:
label_names = dataset["train"].features["label"].names

label_counts = df_train["label"].value_counts().sort_index()

distribution = pd.DataFrame({
    "Label": range(len(label_names)),
    "Kategori": label_names,
    "Jumlah Data": label_counts.values
})

distribution["Persentase (%)"] = (
    distribution["Jumlah Data"] /
    len(df_train) * 100
).round(2)

display(distribution)

**Visualisasi Distribusi Kelas**

Diagram batang menampilkan jumlah artikel per kategori berdasarkan tabel sebelumnya. Visualisasi ini memudahkan pembacaan keseimbangan kelas secara sekilas.

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=distribution,
    x="Kategori",
    y="Jumlah Data"
)

plt.title("Distribusi Kategori Berita AG News")
plt.xlabel("Kategori Berita")
plt.ylabel("Jumlah Artikel")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

**Statistik Panjang Artikel (Jumlah Kata)**

Kolom `word_count` dibuat dengan memecah teks berdasarkan spasi, kemudian `describe()` menghitung statistik deskriptif seperti rata-rata, median, minimum, dan maksimum. Informasi ini menjadi dasar awal penentuan panjang maksimum input model.

In [ ]:
df_train["word_count"] = df_train["text"].apply(
    lambda text: len(str(text).split())
)

print("Statistik Panjang Artikel:")

display(df_train["word_count"].describe())

**Histogram Panjang Artikel**

Histogram dengan kurva KDE memperlihatkan sebaran jumlah kata per artikel. Bentuk sebaran membantu menilai apakah sebagian besar artikel cukup pendek sehingga pemotongan (*truncation*) tidak banyak menghilangkan informasi.

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df_train,
    x="word_count",
    bins=50,
    kde=True
)

plt.title("Distribusi Panjang Artikel Berita AG News")
plt.xlabel("Jumlah Kata")
plt.ylabel("Frekuensi")

plt.tight_layout()
plt.show()

**Pemeriksaan Kualitas Data**

Sel ini memeriksa tiga hal pada data latih dan data uji: nilai kosong (*missing values*), teks yang hanya berisi spasi atau kosong, dan baris duplikat. Pemeriksaan ini memastikan data bersih sebelum dipakai untuk pelatihan dan evaluasi.

In [ ]:
# Memeriksa nilai kosong
print("Missing Values Training:")
print(df_train.isnull().sum())

print("\nMissing Values Testing:")
print(df_test.isnull().sum())

# Memeriksa teks kosong
empty_train = df_train["text"].astype(str).str.strip().eq("").sum()
empty_test = df_test["text"].astype(str).str.strip().eq("").sum()

print("\nTeks Kosong:")
print("Training:", empty_train)
print("Testing :", empty_test)

# Memeriksa duplikasi
print("\nData Duplikat:")
print("Training:", df_train.duplicated().sum())
print("Testing :", df_test.duplicated().sum())

**Contoh Artikel Tiap Kategori**

Untuk setiap kategori, diambil satu artikel pertama dari data latih dan ditampilkan. Hal ini memberi gambaran karakteristik teks pada masing-masing kelas.

In [ ]:
for label_id, category in enumerate(label_names):

    sample = df_train[
        df_train["label"] == label_id
    ].iloc[0]

    print(f"\nKategori: {category}")
    print("Artikel:")
    print(sample["text"])
    print("-" * 100)

**Pembagian Data Latih dan Validasi**

Data latih dibagi menjadi 90% untuk pelatihan dan 10% untuk validasi menggunakan `train_test_split` dengan `seed=42` agar hasil dapat direproduksi. Parameter `stratify_by_column="label"` menjaga proporsi kelas tetap sama pada kedua bagian. Test set resmi dipertahankan terpisah dan tidak dipakai dalam pelatihan.

In [ ]:
# Membagi training set menjadi training dan validation
split_dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column="label"
)

# Membentuk dataset akhir
final_dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
    "test": dataset["test"]
})

# Menampilkan jumlah data
print(final_dataset)

**Verifikasi Jumlah Data Setelah Pembagian**

Jumlah data pada split `train`, `validation`, dan `test` dicetak untuk memastikan pembagian berjalan sesuai rencana. Jumlah ketiganya harus konsisten dengan ukuran data sebelum dibagi.

In [ ]:
print("Jumlah Data Setelah Pembagian:")

for split_name in ["train", "validation", "test"]:

    print(
        f"{split_name.capitalize():12}: "
        f"{len(final_dataset[split_name]):,} data"
    )

**Verifikasi Distribusi Kelas Setiap Split**

Fungsi `Counter` menghitung jumlah data per kategori pada ketiga split. Hasilnya digunakan untuk memastikan stratifikasi berhasil, yaitu proporsi kelas pada data latih dan validasi tetap seimbang.

In [ ]:
for split_name in ["train", "validation", "test"]:

    print(f"\nDistribusi {split_name.upper()}")

    labels = final_dataset[split_name]["label"]
    counts = Counter(labels)

    for label_id, category in enumerate(label_names):
        print(f"{category:10}: {counts.get(label_id, 0):,}")

**Memuat Tokenizer**

`AutoTokenizer` memuat tokenizer dari checkpoint `google-bert/bert-base-uncased`, yaitu tokenizer WordPiece yang mengubah teks menjadi potongan subword dan ID numerik. Ukuran vocabulary dicetak untuk memverifikasi tokenizer termuat. Catatan: checkpoint yang tertulis di kode adalah BERT, bukan DistilBERT, sehingga perlu disesuaikan dengan judul proyek.

In [ ]:
from transformers import AutoTokenizer

# Menentukan model checkpoint
model_checkpoint = "google-bert/bert-base-uncased"

# Memuat tokenizer BERT
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print("Tokenizer berhasil dimuat.")
print("Model:", model_checkpoint)
print("Vocabulary Size:", tokenizer.vocab_size)

**Contoh Proses Tokenisasi**

Satu artikel dari data latih ditokenisasi dengan `max_length=128` dan `truncation=True`. Keluaran memperlihatkan `input_ids` (ID token), `attention_mask` (penanda token asli), dan daftar token hasil konversi, sehingga terlihat bagaimana teks dipecah menjadi subword.

In [ ]:
# Mengambil satu contoh artikel
sample_text = final_dataset["train"][0]["text"]

# Melakukan tokenisasi
encoded_text = tokenizer(
    sample_text,
    truncation=True,
    max_length=128
)

print("Teks Asli:")
print(sample_text)

print("\nInput IDs:")
print(encoded_text["input_ids"])

print("\nAttention Mask:")
print(encoded_text["attention_mask"])

print("\nToken:")
print(tokenizer.convert_ids_to_tokens(encoded_text["input_ids"]))

**Analisis Panjang Token**

Panjang token dihitung tanpa pemotongan pada sampel hingga 5.000 artikel pertama dari data latih, lalu ditampilkan rata-rata, median, minimum, dan maksimum. Sel ini juga menghitung persentase artikel yang melebihi 128 token untuk menilai apakah `max_length=128` memadai.

In [ ]:
# Mengambil maksimal 5.000 artikel
sample_data = final_dataset["train"].select(
    range(min(5000, len(final_dataset["train"])))
)

# Menghitung panjang token
token_lengths = []

for text in sample_data["text"]:
    encoded = tokenizer(text,truncation=False)
    token_lengths.append(len(encoded["input_ids"]))

token_lengths = np.array(token_lengths)

# Statistik panjang token
print("Rata-rata:", round(token_lengths.mean(), 2))
print("Median:", np.median(token_lengths))
print("Minimum:", token_lengths.min())
print("Maksimum:", token_lengths.max())

# Menghitung artikel yang melebihi 128 token
truncated_count = np.sum(token_lengths > 128)
percentage = (truncated_count / len(token_lengths)) * 100

print("\nArtikel melebihi 128 token:", truncated_count)
print("Persentase:", round(percentage, 2), "%")

**Tokenisasi Seluruh Dataset**

`MAX_LENGTH` ditetapkan 128, dan fungsi `tokenize_function` menerapkan tokenizer pada kolom `text`. Metode `map(batched=True)` memproses data per kelompok agar lebih cepat, dan kolom `text` dihapus karena model hanya memerlukan input numerik.

In [ ]:
# Menentukan panjang maksimum token
MAX_LENGTH = 128

# Fungsi tokenisasi
def tokenize_function(examples):
    return tokenizer(examples["text"],truncation=True,max_length=MAX_LENGTH)

# Tokenisasi seluruh dataset
tokenized_dataset = final_dataset.map(tokenize_function,batched=True,remove_columns=["text"])

print("Tokenisasi berhasil.")
print(tokenized_dataset)

**Pemeriksaan Hasil Tokenisasi**

Satu contoh data hasil tokenisasi ditampilkan beserta `input_ids`, `attention_mask`, label, dan jumlah token. Langkah ini memastikan format data sudah benar dan panjangnya tidak melebihi `MAX_LENGTH`.

In [ ]:
# Mengambil satu contoh hasil tokenisasi
sample_tokenized = tokenized_dataset["train"][0]

print("Input IDs:")
print(sample_tokenized["input_ids"])

print("\nAttention Mask:")
print(sample_tokenized["attention_mask"])

print("\nLabel:")
print(sample_tokenized["label"])

print("\nJumlah Token:")
print(len(sample_tokenized["input_ids"]))

**Menyimpan Dataset Hasil Tokenisasi**

Dataset yang sudah ditokenisasi disimpan ke disk dengan `save_to_disk`. Dengan cara ini proses tokenisasi tidak perlu diulang bila sesi Colab dimulai ulang, selama berkas masih tersimpan.

In [ ]:
# Menyimpan dataset hasil tokenisasi
tokenized_dataset.save_to_disk("/content/ag_news_bert_tokenized")

print("Dataset berhasil disimpan.")

**Memuat Model Klasifikasi**

`AutoModelForSequenceClassification` memuat bobot pra-latih dari checkpoint dan menambahkan lapisan klasifikasi baru dengan `num_labels=4` sesuai jumlah kategori. Lapisan baru ini diinisialisasi acak dan akan dilatih pada proses *fine-tuning*. Seperti pada tokenizer, checkpoint yang digunakan adalah BERT.

In [ ]:
from transformers import AutoModelForSequenceClassification

model_checkpoint = "google-bert/bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint,num_labels=4)

print("Model BERT berhasil dimuat.")

**Persiapan DataLoader**

Kolom `label` diubah menjadi `labels` karena itulah nama yang diharapkan model untuk menghitung loss, dan format data diatur menjadi tensor PyTorch. `DataCollatorWithPadding` melakukan *dynamic padding*, yaitu menyamakan panjang sekuens hanya dalam satu batch sehingga lebih efisien. Tiga `DataLoader` dibuat untuk data latih (diacak), validasi, dan uji dengan `BATCH_SIZE = 8`.

In [ ]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

# Mengubah nama kolom label menjadi labels
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# Mengatur format tensor PyTorch
tokenized_dataset.set_format(type="torch",columns=["input_ids", "attention_mask", "labels"])

# Dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

BATCH_SIZE = 8

train_dataloader = DataLoader(
    tokenized_dataset["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_dataset["validation"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    tokenized_dataset["test"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

print("Training batches:", len(train_dataloader))
print("Validation batches:", len(val_dataloader))
print("Testing batches:", len(test_dataloader))

**Konfigurasi Pelatihan**

Perangkat dipilih otomatis (GPU bila tersedia, CPU bila tidak) dan model dipindahkan ke perangkat tersebut. Hiperparameter ditetapkan: 3 epoch dan *learning rate* 2e-5, dengan optimizer `AdamW`. Nilai ini umum dipakai pada fine-tuning model Transformer.

In [ ]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(),lr=LEARNING_RATE)

print("Device:", device)
print("Epochs:", EPOCHS)
print("Learning Rate:", LEARNING_RATE)
print("Batch Size:", BATCH_SIZE)

**Proses Fine-tuning dan Validasi**

Pada setiap epoch, model dilatih dengan siklus `zero_grad`, *forward pass*, `backward` (menghitung gradien), dan `step` (memperbarui bobot). Setelah itu model dievaluasi pada data validasi dalam mode `eval` tanpa gradien. Rata-rata training loss dan validation loss disimpan di `training_history` untuk analisis kurva pembelajaran.

In [ ]:
from tqdm.auto import tqdm

training_history = []

for epoch in range(EPOCHS):

    # Training
    model.train()
    total_train_loss = 0

    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}"):

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_dataloader)

    # Validation
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in val_dataloader:

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)
            total_val_loss += outputs.loss.item()

    avg_val_loss = total_val_loss / len(val_dataloader)

    training_history.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss
    })

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")

**Menyimpan Model Hasil Fine-tuning**

Bobot model dan tokenizer disimpan ke folder `/content/ag_news_bert_model` dengan `save_pretrained`. Folder di `/content` bersifat sementara, sehingga sebaiknya disalin ke Google Drive agar tidak hilang saat sesi berakhir.

In [ ]:
save_path = "/content/ag_news_bert_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model dan tokenizer berhasil disimpan.")

4.1 Memuat Model Hasil Fine-tuning

Model dimuat kembali dari folder penyimpanan untuk membuktikan bahwa hasil fine-tuning dapat dipakai ulang. Model dipindahkan ke perangkat dan diatur ke mode `eval` agar siap untuk inferensi.

In [ ]:
from transformers import AutoModelForSequenceClassification

save_path = "/content/ag_news_bert_model"

model = AutoModelForSequenceClassification.from_pretrained(save_path)
model.to(device)

model.eval()

print("Model hasil fine-tuning berhasil dimuat.")

4.2 Melakukan Prediksi pada Data Testing

Model memprediksi seluruh data uji per batch tanpa menghitung gradien. Kelas dengan skor tertinggi dipilih melalui `argmax` pada *logits*, lalu prediksi dan label asli dikumpulkan dalam daftar. Jumlah keduanya dicetak dan harus sama dengan jumlah data uji.

In [ ]:
import torch
import numpy as np

all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
    for batch in test_dataloader:
        batch = { key: value.to(device) for key, value in batch.items()}
        labels = batch["labels"]

        outputs = model(input_ids=batch["input_ids"],attention_mask=batch["attention_mask"])
        predictions = torch.argmax(outputs.logits,dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Jumlah data testing:", len(all_labels))
print("Jumlah hasil prediksi:", len(all_predictions))

4.3 Menghitung Accuracy, Precision, Recall, dan F1-score

Accuracy, precision, recall, dan F1-score dihitung dengan membandingkan label asli dan prediksi. Parameter `average="weighted"` merata-ratakan metrik antarkelas dengan bobot sesuai jumlah data tiap kelas.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

accuracy = accuracy_score(all_labels, all_predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    average="weighted"
)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-score :", round(f1, 4))

4.4 Menampilkan Classification Report

Classification report menampilkan precision, recall, F1-score, dan *support* untuk setiap kategori. Dari sini terlihat kategori mana yang paling baik maupun paling sulit diprediksi oleh model.

In [ ]:
from sklearn.metrics import classification_report

class_names = ["World", "Sports", "Business", "Sci/Tech"]

print(classification_report(all_labels,all_predictions,target_names=class_names,digits=4))

4.5 Membuat Confusion Matrix

Confusion matrix menampilkan jumlah prediksi benar (diagonal) dan salah (di luar diagonal) untuk setiap pasangan kelas. Visualisasi ini membantu mengidentifikasi pola kesalahan, misalnya kategori yang sering tertukar.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(all_labels,all_predictions)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=class_names)

fig, ax = plt.subplots(figsize=(8, 6))

disp.plot(cmap="Blues",values_format="d",ax=ax)

plt.title("Confusion Matrix - BERT Base Uncased")
plt.show()

Pengujian Berita Manual

Fungsi `predict_news` menerima satu teks berita, menokenisasinya dengan `max_length=128`, lalu memindahkannya ke perangkat yang sama dengan model. Logits keluaran model diubah menjadi probabilitas dengan `softmax`, dan kategori dengan probabilitas tertinggi dipilih sebagai prediksi. Fungsi ini menampilkan kategori hasil prediksi, nilai *confidence*, serta probabilitas untuk setiap kategori.

In [ ]:
import torch

# Daftar kategori berita
labels = ["World", "Sports", "Business", "Sci/Tech"]

def predict_news(news_text):

    model.eval()

    # Tokenisasi teks berita
    inputs = tokenizer(news_text,return_tensors="pt",truncation=True,max_length=128)

    # Memindahkan input ke device
    inputs = {key: value.to(device)for key, value in inputs.items()}

    # Melakukan prediksi
    with torch.no_grad():
        outputs = model(**inputs)

        probabilities = torch.softmax(outputs.logits,dim=1)

    # Mengambil kategori dengan probabilitas tertinggi
    predicted_class = torch.argmax(probabilities,dim=1).item()

    confidence = probabilities[0][predicted_class].item()

    # Menampilkan hasil prediksi
    print("HASIL PREDIKSI BERITA")
    print("=" * 40)
    print("Isi Berita:")
    print(news_text)

    print("\nKategori Prediksi:", labels[predicted_class])
    print(f"Confidence Score: {confidence * 100:.2f}%")

    print("\nProbabilitas Setiap Kategori:")
    for i, label in enumerate(labels):
        score = probabilities[0][i].item() * 100
        print(f"{label}: {score:.2f}%")

Sel ini mencoba fungsi `predict_news` pada satu contoh artikel berita bahasa Inggris yang ditulis manual, bukan berasal dari dataset. Tujuannya mendemonstrasikan bahwa model hasil fine-tuning dapat dipakai untuk memprediksi berita baru. Teks pada variabel `news` dapat diganti dengan berita lain untuk pengujian berikutnya.

In [ ]:
news = """
International rescue teams have arrived in the disaster zone to distribute food, medical supplies, and temporary shelters to thousands of residents displaced by yesterday's 7.2 magnitude earthquake.
"""

predict_news(news)